# GRPO Training — PHI De-identification

Trains the SFT-initialised Qwen3 model using Group Relative Policy Optimization (GRPO).
Reward signal is fully computational — no LLM judge, no GPU-heavy verifier.

**Reward cascade (sequential gate):**
1. `structure`      — valid `<phi_entities>` XML? (hard gate -> 0 if fails)
2. `presence`       — do tagged entities appear in the source note?
3. `completeness`   — entity-level F1 vs ground truth
4. `qc_calibration` — does model's `<qc_status>` tag match actual completeness?
5. `length_penalty` — penalise redundant / over-long outputs


## 0 — Setup

In [ ]:
# install / upgrade key deps if running fresh
# %pip install -q trl>=0.23.0 transformers>=4.45 datasets wandb peft liger-kernel

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ""))  # repo root on path

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from pipeline.rl import GRPOConfig
from pipeline.rl.data import build_grpo_dataset
from pipeline.rl.grpo import build_grpo_trainer, prepare_grpo_reward_function
from pipeline.rl.evaluation import evaluate_policy, compute_metrics
from pipeline.training import expand_tokenizer

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1 — Configuration

Adjust paths and hyper-parameters here.

In [ ]:
# ── model ──────────────────────────────────────────────────────────────────
SFT_MODEL_ID   = "BardiaKh/EdgeDeID-Qwen3-0.6B-SFT"  # HF repo of the SFT checkpoint
OUTPUT_DIR     = "./checkpoints/grpo-qwen3-0.6b"

# ── dataset ────────────────────────────────────────────────────────────────
DATASET_ID     = "BardiaKh/EdgeDeID-Dataset"          # HF dataset repo
TRAIN_SPLIT    = "train"
EVAL_SPLIT     = "validation"                         # set None if no validation split
NOTE_KEY       = "text"                               # column name for the raw note
GT_KEY         = "phi_entities"                       # column name for the ground-truth XML

# ── training ───────────────────────────────────────────────────────────────
config = GRPOConfig(
    model_name              = SFT_MODEL_ID,
    output_dir              = OUTPUT_DIR,
    run_name                = "grpo-edgedeid-qwen3-0.6b",
    wandb_project           = None,             # set None to disable wandb
    learning_rate           = 1e-5,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    num_generations         = 8,                      # completions per prompt per step
    max_prompt_length       = 2048,
    max_completion_length   = 1024,
    seed                    = 2025,
    reward_weights          = {                       # weights for combined reward
        "presence":     0.35,
        "completeness": 0.35,
        "qc":           0.15,
        "length":       0.15,
    },
)

print(config)

## 2 — Load Model & Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID, trust_remote_code=True)
tokenizer = expand_tokenizer(tokenizer)   # adds PHI XML tag tokens

model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.resize_token_embeddings(len(tokenizer))

print(f"Parameters: {model.num_parameters():,}")

### (Optional) Wrap with PEFT / LoRA for memory-efficient GRPO

In [ ]:
USE_LORA = False   # set True on low-VRAM machines

if USE_LORA:
    from peft import LoraConfig, get_peft_model, TaskType
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        target_modules="all-linear",
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

## 3 — Prepare Dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET_ID)
print(raw)

In [ ]:
train_records = build_grpo_dataset(
    raw[TRAIN_SPLIT],
    note_key=NOTE_KEY,
    ground_truth_key=GT_KEY,
)
train_dataset = Dataset.from_list(train_records)

eval_dataset = None
if EVAL_SPLIT and EVAL_SPLIT in raw:
    eval_records = build_grpo_dataset(
        raw[EVAL_SPLIT],
        note_key=NOTE_KEY,
        ground_truth_key=GT_KEY,
    )
    eval_dataset = Dataset.from_list(eval_records)

print(f"Train: {len(train_dataset):,} samples | Eval: {len(eval_dataset) if eval_dataset else 0:,} samples")
print("\nSample prompt:\n", train_dataset[0]["prompt"])

## 4 — Sanity-check Reward Functions

Run the reward pipeline on a single example before launching training.

In [ ]:
from pipeline.rl.rewards import (
    reward_xml_structure,
    reward_phi_text_presence,
    reward_completeness,
    reward_qc_calibration,
    reward_length_penalty,
    compute_grpo_reward,
)

sample = train_dataset[0]
note   = sample["source_note"]
gt     = sample["ground_truth"]

# use the ground truth itself as a perfect-score completion
fake_completion = gt + "\n<qc_status>complete</qc_status>"

print("structure :", reward_xml_structure(fake_completion))
print("presence  :", reward_phi_text_presence(fake_completion, note))
print("completeness:", reward_completeness(fake_completion, gt))
print("qc        :", reward_qc_calibration(fake_completion, reward_completeness(fake_completion, gt)))
print("length    :", reward_length_penalty(fake_completion))
print("combined  :", compute_grpo_reward(fake_completion, note, gt))

## 5 — Build Trainer & Train

In [ ]:
import wandb
if config.wandb_project:
    wandb.init(project=config.wandb_project, name=config.run_name)

In [ ]:
reward_fn = prepare_grpo_reward_function(config)

trainer = build_grpo_trainer(
    model        = model,
    tokenizer    = tokenizer,
    config       = config,
    reward_fn    = reward_fn,
    train_dataset= train_dataset,
    eval_dataset = eval_dataset,
)

trainer.train()

## 6 — Evaluate

In [ ]:
if eval_dataset is not None:
    metrics = evaluate_policy(
        model     = trainer.model,
        tokenizer = tokenizer,
        dataset   = eval_dataset,
        max_new_tokens = 1024,
        temperature    = 0.0,
        batch_size     = 4,
    )
    print("Eval metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
else:
    print("No eval split — skipping evaluation.")

## 7 — Save & Push to HuggingFace

In [ ]:
HF_REPO_ID = "EdgeDeID/Qwen3-0.6B-GRPO"   # target HF repo

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

# push to HF Hub
trainer.model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f"Pushed to {HF_REPO_ID}")

In [ ]:
if config.wandb_project:
    wandb.finish()